Setup (Kevin):
- Share this collection with attendees. Have them open the link and follow along. Provide some context on the collection and what's inside.
- Briefly summarize the workflow we'll go through and what the key takeaways should be.
    - TODO(mengk): figure out what exactly to say here, based on the final form of the demo.

Initial UI exploration (Gregor):
- Show the AgentRun table with all the metadata. Customize it to show the important fields that we'll later download. 
- Click into a single agent run and show that the forecaster structure is logged properly. Point out the transcript group metadata, which captures independent probabilities
- Briefly show the charts feature. Maybe plot the Brier score between the two models.
    - *Aside: This should show that GPT-5 is slightly worse than o3.*
- Segue: What if you wanted to do more advanced data analysis? Download the notebook.

In [ ]:
import IPython
IPython.get_ipython().run_line_magic("load_ext", "autoreload")  # type: ignore
IPython.get_ipython().run_line_magic("autoreload", "2")  # type: ignore

In [ ]:
from docent import Docent
from docent_core._env_util import ENV

DOCENT_API_KEY = ENV.get("DOCENT_API_KEY")
DOCENT_DOMAIN = ENV.get("DOCENT_DOMAIN")
if not DOCENT_DOMAIN or not DOCENT_API_KEY:
    raise ValueError("DOCENT_API_KEY and DOCENT_DOMAIN must be set")
dc = Docent(api_key=DOCENT_API_KEY, domain=DOCENT_DOMAIN)


In [ ]:
COLLECTION_ID = "07287acb-8908-4f4c-95ec-8cdeadcb4423"

In [ ]:
import pandas as pd

pd.set_option('display.float_format', '{:.3f}'.format)

df = dc.dql_result_to_df_experimental(
    dc.execute_dql(
        COLLECTION_ID,
        """
SELECT
    a.id AS agent_run_id,
    a.metadata_json ->> 'experiment_id' AS experiment_id,
    a.metadata_json ->> 'base_llm_name' AS llm,
    a.metadata_json ->> 'llm_reasoning_effort' AS reasoning_effort,
    a.metadata_json ->> 'llm_verbosity' AS verbosity,
    a.metadata_json ->> 'pre_consistency_mean_probability' AS pre_consistency_mean_prob,
    a.metadata_json ->> 'pre_platt_scaling_probability' AS consistency_prob,
    a.metadata_json ->> 'post_platt_scaling_probability' AS scaled_prob,
    a.metadata_json ->> 'final_probability' AS aia_final_prob,
    a.metadata_json ->> 'brier_score' AS aia_brier_score,
    a.metadata_json ->> 'resolved_to' AS gold_prob,
    a.metadata_json ->> 'question' AS question,
    a.metadata_json ->> 'as_of_date' AS as_of_date,
FROM agent_runs a
WHERE a.metadata_json ->> 'finished' = 'true'
GROUP BY a.id
""",
    )
)

df

Gregor:
- *Aside: Everything up to this point should be auto-generated by the notebook.*
- *Aside: All the cells below here can be vibe-coded. You can just use Command-K to tell Cursor what you want it to do because it can see the schema of the data frame from the DQL query.*
- Now let's compute some derived values, namely: the final Brier score (scaled), the score after consistency but before scaling (consistency), and the score before consistency (mean)

In [ ]:
df["scaled_brier_score"] = (df["scaled_prob"] - df["gold_prob"]) ** 2
df["consistency_brier_score"] = (df["consistency_prob"] - df["gold_prob"]) ** 2
df["mean_brier_score"] = (df["pre_consistency_mean_prob"] - df["gold_prob"]) ** 2
df.head()

Gregor:
- We can easily reproduce simple plots, like comparing Brier scores across models. 
- *Aside: Try things out to figure out a prompt that works for you. I also like to tell it to add the numbers above the bars*

In [ ]:
import matplotlib.pyplot as plt

# Group by LLM and calculate mean Brier scores
llm_scores = df.groupby("llm").agg({
    "scaled_brier_score": "mean",
    "consistency_brier_score": "mean",
    "mean_brier_score": "mean"
}).reset_index()

# Create bar chart
x = range(len(llm_scores))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar([i - width for i in x], llm_scores["scaled_brier_score"], width, label="Scaled Brier Score")
bars2 = ax.bar([i for i in x], llm_scores["consistency_brier_score"], width, label="Consistency Brier Score")
bars3 = ax.bar([i + width for i in x], llm_scores["mean_brier_score"], width, label="Mean Brier Score")

# Add value labels above each bar
for bar in bars1:
    height = bar.get_height()
    ax.annotate(f'{height:.3f}',
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3),  # 3 points vertical offset
                textcoords="offset points",
                ha='center', va='bottom', fontsize=8)

for bar in bars2:
    height = bar.get_height()
    ax.annotate(f'{height:.3f}',
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3),  # 3 points vertical offset
                textcoords="offset points",
                ha='center', va='bottom', fontsize=8)

for bar in bars3:
    height = bar.get_height()
    ax.annotate(f'{height:.3f}',
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3),  # 3 points vertical offset
                textcoords="offset points",
                ha='center', va='bottom', fontsize=8)

ax.set_xlabel("LLM")
ax.set_ylabel("Brier Score")
ax.set_title("Scaled vs Consistency vs Mean Brier Score by LLM")
ax.set_xticks(x)
ax.set_xticklabels(llm_scores["llm"], rotation=45, ha="right")
ax.legend()

plt.tight_layout()
plt.show()


Gregor:
- *Aside: GPT-5 seems worse than o3. Additionally, consistency analysis helps*
- One natural question is, what's explaining the discrepancy in performance between the two models? 
- *Aside: Ask Cursor to make a pivot table where it groups by question, and for each question, there is one column for each LLM. Use the scaled_brier_score. Also ask it for an additional column with the diff, then sort by that diff.*

In [ ]:
# Group by question and compute average Brier scores for each model
question_scores = df.pivot_table(
    index="question",
    columns="llm",
    values="scaled_brier_score",
    aggfunc="mean"
).reset_index()

# Get agent_run_ids for each question and model
agent_run_ids = df.groupby(["question", "llm"])["agent_run_id"].apply(list).unstack()
agent_run_ids.columns = [f"{col}_run_ids" for col in agent_run_ids.columns]
agent_run_ids = agent_run_ids.reset_index()

# Merge the agent_run_ids with question_scores
question_scores = question_scores.merge(agent_run_ids, on="question", how="left")

# Create a column for the difference between GPT-5 and O3
question_scores["gpt5_minus_o3"] = question_scores["gpt-5"] - question_scores["o3"]

question_scores = question_scores.dropna(subset=["gpt-5", "o3"]).sort_values("gpt5_minus_o3", ascending=False)
question_scores


Gregor:
- We can also ask for a histogram to see the distribution of differences. 

In [ ]:
# Create histogram of gpt5_minus_o3 differences
fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(question_scores["gpt5_minus_o3"], bins=30, edgecolor="black", alpha=0.7)
ax.set_xlabel("GPT-5 - O3 (Scaled Brier Score Difference)")
ax.set_ylabel("Frequency")
ax.set_title("Distribution of GPT-5 vs O3 Scaled Brier Score Differences")
ax.axvline(x=0, color="red", linestyle="--", label="No difference")
ax.legend()
plt.tight_layout()
plt.show()


Gregor:
- Now comes the time to understand why. Here is where we'd like to demonstrate the multi-run chat. Let's pick an example where o3 consistently did well, but GPT-5 did not. What might explain the difference? 
- *Aside: `iloc` is how you select a row by index. Because the pivot table is sorted by difference, the first one should be the one with the largest average discrepancy.*

In [ ]:
question_scores.iloc[0].to_dict()

Gregor:
- *Aside: If you don't already have columns with the corresponding AgentRun IDs in the pivot table, you need to go back and add them. That way, you can select some AgentRun IDs to put in the multi-run chat.*
- Let's navigate to https://docent-bridgewater.transluce.org/dashboard/07287acb-8908-4f4c-95ec-8cdeadcb4423/chat.
- *Aside: Make sure to use Gemini 3 Pro medium reasoning; it's the only thing with a long-enough ctx window*

Gregor:
- Let's create a couple chats with the first three questions. (*aside: let's say 3; spawn them all at once*)
    - Note: right now, this is a highly manual process, but we're working on a UI for scaling this up across as many pairs of transcripts as you need. For example, we used it to explain performance regressions on TerminalBench.
- Analyze three things in detail, leave comments

Kevin:
- Now, let's convert these annotations into a rubric
- Now, let's spot check results and label things we think are right / wrong
- [Optional] how to optimize the judge
- Connecting it back to quantitative analysis: does it explain the differences?